# CESM2 NAO

In [1]:
import os
import sys
import yaml
import copy

import numpy as np
import pandas as pd
import xarray as xr

/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.4.0' currently installed).
  from pandas.core import (


In [2]:
sys.path.insert(0, os.path.realpath('../libs/'))
import graph_utils as gu
#import verif_utils as vu

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
from eofs.xarray import Eof
import xeofs as xe

In [5]:
import xeofs.data_container.data_container as dc

In [6]:
import pickle
print("xeofs =", xe.__version__)
print("xarray =", xr.__version__)

xeofs = 3.0.4
xarray = 2026.2.0


In [7]:
n_modes_eof = 20
n_modes_rotate = 10

In [10]:
list_nao = []
list_nao_mon = []

for ilead in range(10):
    list_ds = []

    for year in range(1958, 2020):
        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year}-11-01_daily_ensemble.zarr'
        ds = xr.open_zarr(fn)[['Z500']]
        ds = ds.mean('member')
        
        year_pick = year + ilead + 1
        ds = ds.sel(time=slice(f'{year_pick}-01-01', f'{year_pick}-12-31'))
        list_ds.append(ds)

    ds_all = xr.concat(list_ds, dim='time').sortby('time')
    
    slp = ds_all['Z500'] * 9.8
    
    slp = slp.resample(time="MS").mean()

    # convert lon to [-180, 180)
    slp = slp.assign_coords(
        lon=((slp.lon + 180) % 360) - 180
    ).sortby("lon")

    # robust lat slicing
    if slp.lat[0] < slp.lat[-1]:
        slp_na = slp.sel(lat=slice(0, 90)) # , lon=slice(-90, 40)
    else:
        slp_na = slp.sel(lat=slice(90, 0)) # , lon=slice(-90, 40)

    # monthly anomaly
    slp_mon_clim = slp_na.groupby("time.month").mean("time")
    anom = slp_na.groupby("time.month") - slp_mon_clim
    anom = anom.transpose("time", "lat", "lon")
    
    # =================================================== #
    # fit new model
    
    # eof_model = xe.single.EOF(
    #     n_modes=n_modes_eof,
    #     standardize=False,
    #     use_coslat=True,
    # )
    # eof_model.fit(anom, dim="time")

    # rotator = xe.single.EOFRotator(n_modes=n_modes_rotate)
    # rotator.fit(eof_model)
    
    # reof = rotator.components()
    # rpc = rotator.scores(normalized=False)
    
    # # first rotated mode
    # reof1 = reof.isel(mode=0)
    # pc1 = pc1_reof = rpc.isel(mode=0)

    # =================================================== #
    # load existing model
    _original_setitem = dc.DataContainer.__setitem__
    
    def _safe_setitem(self, key, value):
        if not hasattr(self, "_allow_compute"):
            self._allow_compute = {}
        _original_setitem(self, key, value)
    
    dc.DataContainer.__setitem__ = _safe_setitem
    
    # Now load works
    with open("eof_model.pkl", "rb") as f:
        eof_model = pickle.load(f)
    
    with open("reof_model.pkl", "rb") as f:
        rotator = pickle.load(f)
    
    # Restore original method
    dc.DataContainer.__setitem__ = _original_setitem
        
    pc_new = eof_model.transform(anom)
    rpc_new = rotator.transform(anom)
    
    # Get individual modes
    pc1 = pc_new.isel(mode=0)
    pc1 = rpc1 = rpc_new.isel(mode=0)
    
    # sign convention using simple monthly box NAO
    if slp.lat[0] < slp.lat[-1]:
        south = slp.sel(lat=slice(35, 40), lon=slice(-35, -20)).mean(("lat", "lon"))
        north = slp.sel(lat=slice(60, 70), lon=slice(-30, -10)).mean(("lat", "lon"))
    else:
        south = slp.sel(lat=slice(40, 35), lon=slice(-35, -20)).mean(("lat", "lon"))
        north = slp.sel(lat=slice(70, 60), lon=slice(-30, -10)).mean(("lat", "lon"))

    nao_box = south - north
    nao_box = nao_box.groupby("time.month") - nao_box.groupby("time.month").mean("time")

    corr = float(xr.corr(pc1, nao_box))
    print(corr)
    
    if corr < 0:
        pc1 = -pc1

    # pc1 is already monthly
    nao_monthly = (pc1 - pc1.mean("time")) / pc1.std("time")
    nao_monthly = nao_monthly.compute()
    
    nao_monthly.name = "nao_monthly_eof"
    ds_monthly = nao_monthly.to_dataset(name="NAO monthly")
    
    # annual
    nao_ann = nao_monthly.groupby("time.year").mean("time")
    nao_ann.name = "nao_ann_eof"

    # DJF
    years = np.asarray(np.unique(nao_monthly["time.year"].values))
    first_year = int(years.min())
    
    pieces = []
    jf_first = nao_monthly.where(
        (nao_monthly["time.year"] == first_year) &
        (nao_monthly["time.month"].isin([1, 2])),
        drop=True
    )

    if jf_first.sizes.get("time", 0) == 2:
        jf_first_mean = jf_first.mean("time")
        jf_first_mean = jf_first_mean.expand_dims(year=[first_year])
        pieces.append(jf_first_mean)
        
    djf_raw = (
        nao_monthly.where(nao_monthly["time.month"].isin([12, 1, 2]), drop=True)
                   .resample(time="QS-DEC")
    )

    djf = djf_raw.mean()
    djf_count = djf_raw.count()

    djf = djf.where(djf_count == 3, drop=True)
    djf = djf.where(djf["time.month"] == 12, drop=True)
    djf = djf.assign_coords(year=djf["time.year"] + 1)
    djf = djf.swap_dims({"time": "year"}).drop_vars("time")
    djf = djf.dropna("year")

    # avoid duplicate first year if it somehow exists
    djf = djf.where(djf["year"] != first_year, drop=True)
    pieces.append(djf)

    nao_djf = xr.concat(pieces, dim="year").sortby("year")
    nao_djf.name = "nao_djf_eof"
    

    # other seasons
    nao_mam = nao_monthly.where(nao_monthly["time.month"].isin([3, 4, 5]), drop=True).groupby("time.year").mean("time")
    nao_jja = nao_monthly.where(nao_monthly["time.month"].isin([6, 7, 8]), drop=True).groupby("time.year").mean("time")
    nao_son = nao_monthly.where(nao_monthly["time.month"].isin([9, 10, 11]), drop=True).groupby("time.year").mean("time")

    nao_mam.name = "nao_mam_eof"
    nao_jja.name = "nao_jja_eof"
    nao_son.name = "nao_son_eof"

    ds_nao_yearly = xr.Dataset({
        "nao_ann_eof": nao_ann,
        "nao_djf_eof": nao_djf,
        "nao_mam_eof": nao_mam,
        "nao_jja_eof": nao_jja,
        "nao_son_eof": nao_son,
    })
    
    ds_nao_yearly = ds_nao_yearly.rename({'year': 'init_year'})
    ds_nao_yearly = ds_nao_yearly.assign_coords({'init_year': np.arange(1958, 2020)})
    
    list_nao.append(ds_nao_yearly)

    if ilead == 0:
        time_val = ds_monthly['time'].values
    else:
        ds_monthly['time'] = time_val
    
    list_nao_mon.append(ds_monthly)
    
    print(f'lead year {ilead + 1} done')
    
ds_nao_all = xr.concat(
    list_nao,
    dim=xr.IndexVariable("lead_year", np.arange(10))
)

ds_nao_mon = xr.concat(
    list_nao_mon,
    dim=xr.IndexVariable("lead_year", np.arange(10))
)

/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)
/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)


-0.6898875298676772
lead year 1 done


/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)
/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)


-0.6900808432752568
lead year 2 done


/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)
/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)


...

-0.6253520140833085
lead year 8 done


/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)
/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)


-0.6181172475079717
lead year 9 done


/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)
/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/xeofs/preprocessing/multi_index_converter.py:49: FutureWarning: Deleting a single level of a MultiIndex is deprecated. Previously, this deleted all levels of a MultiIndex. Please also drop the following variables: {'dim1', 'dim2'} to avoid an error in the future.
  X_transformed = X_transformed.drop_vars(dim)


-0.6593741299806402
lead year 10 done


In [11]:
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/NAO_CESM_REOF.zarr'

ds_nao_all = ds_nao_all.chunk({'init_year': 62, 'lead_year': 10})
ds_nao_all.to_zarr(save_name, mode='w')

In [12]:
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/NAO_CESM_REOF.zarr'

ds_nao_all = ds_nao_all.chunk({'init_year': 744, 'lead_year': 10})
ds_nao_all.to_zarr(save_name, mode='w')

In [13]:
ds_nao_all

<xarray.Dataset> Size: 25kB
Dimensions:      (lead_year: 10, init_year: 62)
Coordinates:
  * lead_year    (lead_year) int64 80B 0 1 2 3 4 5 6 7 8 9
  * init_year    (init_year) int64 496B 1958 1959 1960 1961 ... 2017 2018 2019
    mode         int64 8B 1
Data variables:
    nao_ann_eof  (lead_year, init_year) float64 5kB dask.array<chunksize=(10, 62), meta=np.ndarray>
    nao_djf_eof  (lead_year, init_year) float64 5kB dask.array<chunksize=(10, 62), meta=np.ndarray>
    nao_mam_eof  (lead_year, init_year) float64 5kB dask.array<chunksize=(10, 62), meta=np.ndarray>
    nao_jja_eof  (lead_year, init_year) float64 5kB dask.array<chunksize=(10, 62), meta=np.ndarray>
    nao_son_eof  (lead_year, init_year) float64 5kB dask.array<chunksize=(10, 62), meta=np.ndarray>

### Monthly only

In [13]:
list_nao = []

for ilead in range(10):
    list_ds = []

    for year in range(1958, 2020):
        fn = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year}-11-01_daily_ensemble.zarr'
        ds = xr.open_zarr(fn)[['PSL']]
        ds = ds.mean('member')

        year_pick = year + ilead + 1
        ds = ds.sel(time=slice(f'{year_pick}-01-01', f'{year_pick}-12-31'))
        list_ds.append(ds)

    ds_all = xr.concat(list_ds, dim='time').sortby('time')

    # SLP in hPa and convert daily -> monthly
    slp = ds_all['PSL'] / 100.0
    slp = slp.resample(time="MS").mean()

    # convert lon to [-180, 180)
    slp = slp.assign_coords(
        lon=((slp.lon + 180) % 360) - 180
    ).sortby("lon")

    # robust lat slicing
    if slp.lat[0] < slp.lat[-1]:
        slp_na = slp.sel(lat=slice(20, 80), lon=slice(-90, 40))
    else:
        slp_na = slp.sel(lat=slice(80, 20), lon=slice(-90, 40))

    # monthly anomaly
    slp_mon_clim = slp_na.groupby("time.month").mean("time")
    anom = slp_na.groupby("time.month") - slp_mon_clim
    anom = anom.transpose("time", "lat", "lon")
    
    # weights
    w_lat = np.sqrt(np.cos(np.deg2rad(anom["lat"])))
    weights_2d = xr.DataArray(
        np.broadcast_to(w_lat.values[:, None], (anom.sizes["lat"], anom.sizes["lon"])),
        coords={"lat": anom["lat"], "lon": anom["lon"]},
        dims=("lat", "lon"),
    )
    
    # EOF
    solver = Eof(anom, weights=weights_2d)
    eof1 = solver.eofs(neofs=1)[0]
    pc1 = solver.pcs(npcs=1, pcscaling=1)[:, 0]

    # sign convention using simple monthly box NAO
    if slp.lat[0] < slp.lat[-1]:
        south = slp.sel(lat=slice(35, 40), lon=slice(-35, -20)).mean(("lat", "lon"))
        north = slp.sel(lat=slice(60, 70), lon=slice(-30, -10)).mean(("lat", "lon"))
    else:
        south = slp.sel(lat=slice(40, 35), lon=slice(-35, -20)).mean(("lat", "lon"))
        north = slp.sel(lat=slice(70, 60), lon=slice(-30, -10)).mean(("lat", "lon"))

    nao_box = south - north
    nao_box = nao_box.groupby("time.month") - nao_box.groupby("time.month").mean("time")

    corr = xr.corr(pc1, nao_box)
    if float(corr) < 0:
        eof1 = -eof1
        pc1 = -pc1

    # pc1 is already monthly
    nao_monthly = (pc1 - pc1.mean("time")) / pc1.std("time")
    nao_monthly.name = "nao_monthly_eof"
    
    ds_monthly = nao_monthly.to_dataset(name="NAO monthly")

    if ilead == 0:
        time_val = ds_monthly['time'].values
    else:
        ds_monthly['time'] = time_val
        
    list_nao.append(ds_monthly)
    
    print(f'lead year {ilead + 1} done')
    
ds_nao_all = xr.concat(
    list_nao,
    dim=xr.IndexVariable("lead_year", np.arange(10))
)

lead year 1 done
lead year 2 done
lead year 3 done
lead year 4 done
lead year 5 done
lead year 6 done
lead year 7 done
lead year 8 done
lead year 9 done
lead year 10 done


In [ ]:
ds_nao_all

In [17]:
save_name = '/glade/derecho/scratch/ksha/EPRI_data/METRICS/NAO_mon.zarr'

ds_nao_all = ds_nao_all.chunk({'time': 744, 'lead_year': 10})
# ds_nao_all.to_zarr(save_name, mode='w')

In [6]:
ds_monthly = nao_monthly.to_dataset(name="NAO monthly")

In [7]:
ds_monthly

<xarray.Dataset>
Dimensions:      (time: 744)
Coordinates:
  * time         (time) object 1959-01-01 00:00:00 ... 2020-12-01 00:00:00
    mode         int64 0
    month        (time) int64 1 2 3 4 5 6 7 8 9 10 11 ... 3 4 5 6 7 8 9 10 11 12
Data variables:
    NAO monthly  (time) float64 -2.066 -1.37 0.08683 ... -0.4777 1.3 0.9045